# Full CAP-ZW prototype

Research hand-off for the learned model. Proposed training objectives are: clean/attacked pair consistency, collision-aware hard negatives, bit balance, bit decorrelation, differentiable thresholding, and Pareto weighting.

These are candidate components; no component is claimed novel by itself.

In [ ]:
import torch
import torch.nn as nn

class HashNet(nn.Module):
    def __init__(self, nbits=256):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,3,2,1),nn.ReLU(),
            nn.Conv2d(32,64,3,2,1),nn.ReLU(),
            nn.Conv2d(64,128,3,2,1),nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.head=nn.Linear(128,nbits)
    def forward(self,x): return self.head(self.net(x).flatten(1))

def ste_binary(logits):
    hard=(logits>=0).float(); soft=torch.sigmoid(logits)
    return hard + soft - soft.detach()

def collision_aware_objective(logits_clean, logits_attack, labels, margin=0.4):
    b_clean=ste_binary(logits_clean); b_attack=ste_binary(logits_attack)
    robustness=torch.mean(torch.abs(b_clean-b_attack))
    balance=torch.mean((b_clean.mean(0)-0.5)**2)
    z=b_clean-b_clean.mean(0,keepdim=True)
    covariance=(z.T@z)/(z.shape[0]-1+1e-6)
    decorrelation=(covariance-torch.diag(torch.diag(covariance))).abs().mean()
    distances=torch.cdist(b_clean,b_clean,p=1)/b_clean.shape[1]
    same=labels[:,None].eq(labels[None,:])
    negatives=distances.masked_fill(same,float('inf')).min(dim=1).values
    hard_negative=torch.relu(margin-negatives).mean()
    return robustness,balance,decorrelation,hard_negative

model=HashNet(256)
print(model)